In [1]:
import torch
from torch import nn
import polars as pl
import numpy as np

In [ ]:
# ==========================================
# 0 Hyperparameters
# ==========================================

MAX_RUL = 130
WINDOW_SEQ = 30
BATCH_SIZE = 32
HIDDEN_SIZE = 64
NUM_LAYERS = 4
NUM_OF_WORKERS = 0
LR = 1e-3
EPOCHS = 40
WEIGHT_DECAY = 5e-4
T0            = 30         # cosine annealing period
T_MULT        = 2          # cosine annealing multiplier


In [ ]:
# ==========================================
# 1 LOAD DATA
# ==========================================

col_names = ["unit", "cycle"] + [f'op_{i}' for i in range(3)] + [f"s_{i}" for i in range(21)]
drop_cols = ["s_0", "s_4", "s_15", "s_17", "s_18"]
feature_cols = [c for c in col_names if c not in ["unit", "cycle"] + drop_cols]

def load_all_data():

    train_lazy_df = pl.scan_csv(
        "CMAPSSData/train_FD00*.txt",
        separator=" ",
        truncate_ragged_lines=True,
        has_header=False,
        new_columns=col_names,
        include_file_paths="file_path"
    )

    train_df_load = train_lazy_df.select(col_names + ["file_path"]).with_columns(
        (pl.col("cycle").max().over(["file_path", "unit"]) - pl.col("cycle"))
        .clip(upper_bound=MAX_RUL)
        .alias("RUL")
    ).collect()


    test_labels_lazy = pl.scan_csv(
        "CMAPSSData/RUL_FD00*.txt",
        separator=" ",
        truncate_ragged_lines=True,
        has_header=False,
        include_file_paths="file_path"
    ).select([
        pl.col("file_path"),
        # Generujemy numer silnika (1, 2, 3...) dla każdego pliku z osobna
        pl.int_range(1, pl.len() + 1).over("file_path").alias("unit"),
        pl.col("column_1").alias("true_end_rul")
    ]).collect()


    test_lazy_df = pl.scan_csv(
        "CMAPSSData/test_FD00*.txt",
        separator=" ",
        truncate_ragged_lines=True,
        has_header=False,
        new_columns=col_names,
        include_file_paths="file_path"
    ).select(col_names + ["file_path"]).collect()

    return train_df_load, test_lazy_df, test_labels_lazy

In [ ]:
# ==========================================
# 2 Creat windows
# ==========================================

def train_windows(df, feature_cols, window=WINDOW_SEQ):

    X, y = [], []

    for _, group_df in df.group_by(["file_path", "unit"]):
        group_df = group_df.sort("cycle")
        data = group_df[feature_cols].to_numpy()
        labels = group_df["RUL"].to_numpy()
        # print(CMAPSSData.shape, labels.shape)
        for i in range(len(group_df)-window+1):
            X.append(data[i:i+window])
            y.append(labels[i+window-1])
        # print(X.shape, y.shape)

    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)

def test_windows(df, labels, feature_cols, window=WINDOW_SEQ):

    X, y = [], []
    for i, (_, group_df) in enumerate(df.group_by(["file_path", "unit"])):
        group_df  = group_df.sort("cycle")
        data = group_df[feature_cols].to_numpy()
        if len(data) >= window:
            X.append(data[-window:])
        else:
            #pad shorter seq
            pad = np.zeros((window - len(data), len(feature_cols)))
            X.append(np.vstack([pad, data]))
        y.append(labels[i])

    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)

In [ ]:
# ==========================================
# 3 Create Datasets
# ==========================================

from torch.utils.data import Dataset, DataLoader

class Train_Dataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.from_numpy(X).float()
        self.y = torch.from_numpy(y).float()

    def __len__(self):
        return len(self.X)

    def __getitem__(self, item):
        return self.X[item], self.y[item]

class Test_Dataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.from_numpy(X).float()
        self.y = torch.from_numpy(y).float()

    def __len__(self):
        return len(self.X)

    def __getitem__(self, item):
        return self.X[item], self.y[item]


In [ ]:
# ==========================================
# 4 Model
# ==========================================
class GRU(nn.Module):
    def __init__(self, num_layers, input_size, hidden_size):
        super().__init__()
        self.num_layers = num_layers
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.gru = nn.GRU(input_size, hidden_size, num_layers, batch_first=True)

        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, X):
        h0 = torch.zeros(self.num_layers, X.shape[0], self.hidden_size, device=X.device)
        out, hn = self.gru(X, h0)
        return self.fc(out[:, -1, :]).squeeze(-1)

In [ ]:
# ==========================================
# 5 Trainer function
# ==========================================
def MSE(y_hat, y):
    return torch.mean((y_hat - y) ** 2)


class Trainer:
    def __init__(self, model, train_dataloader, test_dataloader, optimizer, scheduler, device, loss_fn, epoch):
        self.model = model
        self.train_dataloader = train_dataloader
        self.test_dataloader = test_dataloader
        self.optimizer = optimizer
        self.scheduler = scheduler
        self.device = device
        self.loss_fn = loss_fn          # e.g., MSE for training
        self.epoch = epoch

    def fit_epoch(self):
        self.model.train()
        total_loss = 0.0
        total_samples = 0
        total_squared_error = 0.0

        for X, y in self.train_dataloader:
            X, y = X.to(self.device), y.to(self.device)

            self.optimizer.zero_grad(set_to_none=True)
            pred = self.model(X).squeeze()   # ensure shape (batch,)
            loss = self.loss_fn(pred, y)     # MSE
            loss.backward()
            self.optimizer.step()

            batch_size = y.size(0)
            total_loss += loss.item() * batch_size          # sum of MSE losses (weighted by batch size)
            total_squared_error += torch.sum((pred - y) ** 2).item()
            total_samples += batch_size

        # Return average MSE and RMSE for the epoch (optional)
        avg_mse = total_loss / total_samples
        rmse = np.sqrt(total_squared_error / total_samples)
        return avg_mse, rmse

    def validate_epoch(self):
        self.model.eval()
        total_squared_error = 0.0
        total_samples = 0

        with torch.no_grad():
            for X, y in self.test_dataloader:
                X, y = X.to(self.device), y.to(self.device)
                pred = self.model(X).squeeze()
                total_squared_error += torch.sum((pred - y) ** 2).item()
                total_samples += y.size(0)

        rmse = np.sqrt(total_squared_error / total_samples)
        return rmse

    def fit(self):
        for epoch in range(self.epoch):
            train_mse, train_rmse = self.fit_epoch()
            val_rmse = self.validate_epoch()
            self.scheduler.step()

            print(f"Epoch {epoch+1}/{self.epoch} | Train MSE: {train_mse:.4f} | Train RMSE: {train_rmse:.4f} | Val RMSE: {val_rmse:.4f}")

In [ ]:
# Load
if __name__ == "__main__":

    train_df, test_df, RUL_test = load_all_data()

    train_df, train_labels = train_windows(train_df, feature_cols)
    test_df, test_labels = test_windows(test_df, RUL_test["true_end_rul"], feature_cols)

    train_dataloader = DataLoader(Train_Dataset(train_df, train_labels), batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_OF_WORKERS, pin_memory=True, drop_last=True)
    test_dataloader = DataLoader(Test_Dataset(test_df, test_labels), batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_OF_WORKERS, pin_memory=True)

    loss_fn = MSE
    GRU = GRU(NUM_LAYERS, len(feature_cols), HIDDEN_SIZE)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    GRU.to(device)
    optimiser = torch.optim.AdamW(
        GRU.parameters(), lr=LR, weight_decay=WEIGHT_DECAY
    )
    # Cosine Annealing with Warm Restarts
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimiser, T_0=T0, T_mult=T_MULT, eta_min=LR * 0.01
    )

    trainer = Trainer(
        model=GRU,
        train_dataloader=train_dataloader,
        test_dataloader=test_dataloader,
        optimizer=optimiser,
        scheduler=scheduler,
        device=device,
        loss_fn=loss_fn,
        epoch=EPOCHS
    )

    trainer.fit()

In [2]:
import torch
from torch import nn
import polars as pl
import numpy as np

from torch.utils.data import Dataset, DataLoader


# ==========================================
# 0 Hyperparameters
# ==========================================

MAX_RUL = 130
WINDOW_SEQ = 30
BATCH_SIZE = 32
HIDDEN_SIZE = 64
NUM_LAYERS = 4

# IMPORTANT for Windows + Jupyter
NUM_OF_WORKERS = 0

LR = 1e-3
EPOCHS = 40
WEIGHT_DECAY = 5e-4
T0 = 30
T_MULT = 2


# ==========================================
# 1 LOAD DATA
# ==========================================

col_names = (
    ["unit", "cycle"]
    + [f"op_{i}" for i in range(3)]
    + [f"s_{i}" for i in range(21)]
)

drop_cols = ["s_0", "s_4", "s_15", "s_17", "s_18"]

feature_cols = [
    c for c in col_names
    if c not in ["unit", "cycle"] + drop_cols
]


def load_all_data():

    # -------------------------------
    # TRAIN
    # -------------------------------
    train_df = (
        pl.scan_csv(
            "CMAPSSData/train_FD00*.txt",
            separator=" ",
            truncate_ragged_lines=True,
            has_header=False,
            new_columns=col_names,
            include_file_paths="file_path",
        )
        .select(col_names + ["file_path"])
        .with_columns(
            (
                pl.col("cycle").max().over(["file_path", "unit"])
                - pl.col("cycle")
            )
            .clip(upper_bound=MAX_RUL)
            .alias("RUL")
        )
        .collect()
    )

    # -------------------------------
    # TEST
    # -------------------------------
    test_df = (
        pl.scan_csv(
            "CMAPSSData/test_FD00*.txt",
            separator=" ",
            truncate_ragged_lines=True,
            has_header=False,
            new_columns=col_names,
            include_file_paths="file_path",
        )
        .select(col_names + ["file_path"])
        .collect()
    )

    # -------------------------------
    # TEST RUL LABELS
    # Each line = one unit
    # -------------------------------
    rul_df = (
        pl.scan_csv(
            "CMAPSSData/RUL_FD00*.txt",
            separator=" ",
            has_header=False,
            truncate_ragged_lines=True,
            include_file_paths="file_path",
        )
        .select([
            pl.col("file_path"),
            pl.col("column_1").cast(pl.Float32).alias("true_end_rul"),
        ])
        .with_columns(
            pl.int_range(
                1,
                pl.len() + 1
            )
            .over("file_path")
            .alias("unit")
        )
        .collect()
    )

    return train_df, test_df, rul_df

from pathlib import Path
import re
import numpy as np
import polars as pl


# ==========================================
# Load test RUL labels
# ==========================================

def load_rul_files(data_dir="CMAPSSData"):

    rows = []

    for path in sorted(Path(data_dir).glob("RUL_FD*.txt")):

        # Example:
        # RUL_FD001.txt -> FD001
        match = re.search(r"(FD\d+)", path.name)

        if match is None:
            raise ValueError(f"Could not determine dataset ID from {path}")

        dataset_id = match.group(1)

        with open(path, "r") as f:
            values = [
                float(line.strip())
                for line in f
                if line.strip()
            ]

        # First line = unit 1
        # Second line = unit 2
        # etc.
        for unit, rul in enumerate(values, start=1):
            rows.append({
                "dataset": dataset_id,
                "unit": unit,
                "true_end_rul": rul,
            })

    return pl.DataFrame(rows)

# ==========================================
# 2 CREATE WINDOWS
# ==========================================

def train_windows(df, feature_cols, window=WINDOW_SEQ):

    X, y = [], []

    for _, group_df in df.group_by(["file_path", "unit"]):
        group_df = group_df.sort("cycle")

        data = group_df.select(feature_cols).to_numpy()
        labels = group_df["RUL"].to_numpy()

        for i in range(len(data) - window + 1):
            X.append(data[i:i + window])
            y.append(labels[i + window - 1])

    return (
        np.asarray(X, dtype=np.float32),
        np.asarray(y, dtype=np.float32),
    )


def test_windows(
    df,
    labels_df,
    feature_cols,
    window=WINDOW_SEQ
):

    X = []
    y = []

    # Make lookup:
    # ("FD001", 1) -> RUL
    # ("FD001", 2) -> RUL
    # ...
    labels_lookup = {
        (row["dataset"], row["unit"]): row["true_end_rul"]
        for row in labels_df.iter_rows(named=True)
    }

    for (file_path, unit), group_df in df.group_by(
        ["file_path", "unit"],
        maintain_order=True
    ):

        group_df = group_df.sort("cycle")

        data = group_df.select(feature_cols).to_numpy()

        # Extract FD001 / FD002 / ...
        filename = Path(file_path).name

        match = re.search(r"(FD\d+)", filename)

        if match is None:
            raise ValueError(
                f"Could not determine dataset ID from {filename}"
            )

        dataset_id = match.group(1)

        # Last WINDOW_SEQ cycles
        if len(data) >= window:

            seq = data[-window:]

        else:

            pad = np.zeros(
                (window - len(data), len(feature_cols)),
                dtype=np.float32
            )

            seq = np.vstack([pad, data])

        X.append(seq)

        key = (dataset_id, unit)

        if key not in labels_lookup:
            raise ValueError(
                f"Missing RUL for dataset={dataset_id}, unit={unit}"
            )

        y.append(labels_lookup[key])

    return (
        np.asarray(X, dtype=np.float32),
        np.asarray(y, dtype=np.float32)
    )


# ==========================================
# 3 DATASETS
# ==========================================

class TrainDataset(Dataset):

    def __init__(self, X, y):
        self.X = torch.from_numpy(X)
        self.y = torch.from_numpy(y)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


class TestDataset(Dataset):

    def __init__(self, X, y):
        self.X = torch.from_numpy(X)
        self.y = torch.from_numpy(y)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


# ==========================================
# 4 MODEL
# ==========================================

class GRUModel(nn.Module):

    def __init__(self, num_layers, input_size, hidden_size):
        super().__init__()

        self.num_layers = num_layers
        self.hidden_size = hidden_size

        self.gru = nn.GRU(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
        )

        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, X):

        h0 = torch.zeros(
            self.num_layers,
            X.size(0),
            self.hidden_size,
            device=X.device,
        )

        out, _ = self.gru(X, h0)

        return self.fc(out[:, -1, :]).squeeze(-1)


# ==========================================
# 5 LOSS
# ==========================================

def mse_loss(y_hat, y):
    return torch.mean((y_hat - y) ** 2)


# ==========================================
# 6 TRAINER
# ==========================================

class Trainer:

    def __init__(
        self,
        model,
        train_dataloader,
        test_dataloader,
        optimizer,
        scheduler,
        device,
        loss_fn,
        epochs,
    ):
        self.model = model
        self.train_dataloader = train_dataloader
        self.test_dataloader = test_dataloader
        self.optimizer = optimizer
        self.scheduler = scheduler
        self.device = device
        self.loss_fn = loss_fn
        self.epochs = epochs

    def fit_epoch(self):

        self.model.train()

        total_squared_error = 0.0
        total_samples = 0

        for X, y in self.train_dataloader:

            X = X.to(self.device, non_blocking=True)
            y = y.to(self.device, non_blocking=True)

            self.optimizer.zero_grad(set_to_none=True)

            pred = self.model(X)

            loss = self.loss_fn(pred, y)

            loss.backward()
            self.optimizer.step()

            batch_size = y.size(0)

            total_squared_error += (
                torch.sum((pred - y) ** 2).item()
            )

            total_samples += batch_size

        mse = total_squared_error / total_samples
        rmse = np.sqrt(mse)

        return mse, rmse

    def validate_epoch(self):

        self.model.eval()

        total_squared_error = 0.0
        total_samples = 0

        with torch.no_grad():

            for X, y in self.test_dataloader:

                X = X.to(self.device, non_blocking=True)
                y = y.to(self.device, non_blocking=True)

                pred = self.model(X)

                total_squared_error += (
                    torch.sum((pred - y) ** 2).item()
                )

                total_samples += y.size(0)

        mse = total_squared_error / total_samples
        rmse = np.sqrt(mse)

        return rmse

    def fit(self):

        for epoch in range(self.epochs):

            train_mse, train_rmse = self.fit_epoch()
            val_rmse = self.validate_epoch()

            self.scheduler.step()

            current_lr = self.optimizer.param_groups[0]["lr"]

            print(
                f"Epoch {epoch + 1:02d}/{self.epochs} | "
                f"Train MSE: {train_mse:.4f} | "
                f"Train RMSE: {train_rmse:.4f} | "
                f"Val RMSE: {val_rmse:.4f} | "
                f"LR: {current_lr:.2e}"
            )


# ==========================================
# 7 MAIN
# ==========================================

train_df, test_df, _ = load_all_data()

RUL_test = load_rul_files()

train_X, train_y = train_windows(
    train_df,
    feature_cols,
    WINDOW_SEQ
)

test_X, test_y = test_windows(
    test_df,
    RUL_test,
    feature_cols,
    WINDOW_SEQ
)
print("Train X:", train_X.shape)
print("Train y:", train_y.shape)
print("Test X :", test_X.shape)
print("Test y :", test_y.shape)

train_dataset = TrainDataset(train_X, train_y)
test_dataset = TestDataset(test_X, test_y)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# pin_memory is useful when transferring CPU tensors to CUDA
use_pin_memory = device.type == "cuda"

train_dataloader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_OF_WORKERS,
    pin_memory=use_pin_memory,
    drop_last=True,
)

test_dataloader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_OF_WORKERS,
    pin_memory=use_pin_memory,
)

model = GRUModel(
    num_layers=NUM_LAYERS,
    input_size=len(feature_cols),
    hidden_size=HIDDEN_SIZE,
).to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY,
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer,
    T_0=T0,
    T_mult=T_MULT,
    eta_min=LR * 0.01,
)

trainer = Trainer(
    model=model,
    train_dataloader=train_dataloader,
    test_dataloader=test_dataloader,
    optimizer=optimizer,
    scheduler=scheduler,
    device=device,
    loss_fn=mse_loss,
    epochs=EPOCHS,
)

trainer.fit()

Train X: (139798, 30, 19)
Train y: (139798,)
Test X : (707, 30, 19)
Test y : (707,)
Epoch 01/40 | Train MSE: 2679.3462 | Train RMSE: 51.7624 | Val RMSE: 51.3616 | LR: 9.97e-04
Epoch 02/40 | Train MSE: 1896.2609 | Train RMSE: 43.5461 | Val RMSE: 51.4366 | LR: 9.89e-04
Epoch 03/40 | Train MSE: 1896.3684 | Train RMSE: 43.5473 | Val RMSE: 51.3111 | LR: 9.76e-04
Epoch 04/40 | Train MSE: 1896.2012 | Train RMSE: 43.5454 | Val RMSE: 51.4673 | LR: 9.57e-04
Epoch 05/40 | Train MSE: 1896.2770 | Train RMSE: 43.5463 | Val RMSE: 51.3059 | LR: 9.34e-04
Epoch 06/40 | Train MSE: 1896.3264 | Train RMSE: 43.5468 | Val RMSE: 51.3945 | LR: 9.05e-04
Epoch 07/40 | Train MSE: 1896.1809 | Train RMSE: 43.5452 | Val RMSE: 51.4257 | LR: 8.73e-04
Epoch 08/40 | Train MSE: 1896.1683 | Train RMSE: 43.5450 | Val RMSE: 51.2989 | LR: 8.36e-04
Epoch 09/40 | Train MSE: 1896.0748 | Train RMSE: 43.5439 | Val RMSE: 51.3414 | LR: 7.96e-04
Epoch 10/40 | Train MSE: 1896.1665 | Train RMSE: 43.5450 | Val RMSE: 51.3692 | LR: 7.53e